In [22]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

In [23]:
# 1-2. Import and explore
df = pd.read_csv("Telecom_Customer_Churn.csv")
print(df.shape)
df.info()
print(df.describe()) 

(7078, 21)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7078 entries, 0 to 7077
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7078 non-null   object 
 1   gender            7078 non-null   object 
 2   SeniorCitizen     7078 non-null   int64  
 3   Partner           7078 non-null   object 
 4   Dependents        7078 non-null   object 
 5   tenure            7078 non-null   int64  
 6   PhoneService      7078 non-null   object 
 7   MultipleLines     7048 non-null   object 
 8   InternetService   7078 non-null   object 
 9   OnlineSecurity    7078 non-null   object 
 10  OnlineBackup      7078 non-null   object 
 11  DeviceProtection  7078 non-null   object 
 12  TechSupport       7078 non-null   object 
 13  StreamingTV       7078 non-null   object 
 14  StreamingMovies   7078 non-null   object 
 15  Contract          7078 non-null   object 
 16  PaperlessBilling  7078 non-null

In [24]:
# 3. Missing Values
print(df.isnull().sum())
for c in df.select_dtypes("number"):
    df[c]=df[c].fillna(df[c].median())

for c in df.select_dtypes("object"):
    df[c] = df[c].fillna(df[c].mode()[0])

customerID           0
gender               0
SeniorCitizen        0
Partner              0
Dependents           0
tenure               0
PhoneService         0
MultipleLines       30
InternetService      0
OnlineSecurity       0
OnlineBackup         0
DeviceProtection     0
TechSupport          0
StreamingTV          0
StreamingMovies      0
Contract             0
PaperlessBilling     0
PaymentMethod       26
MonthlyCharges       0
TotalCharges        40
Churn                0
dtype: int64


In [25]:
# 4. Duplicates
df = df.drop_duplicates()

In [26]:
# 5. Inconsistent data
for c in df.select_dtypes("object"):
    df[c]=df[c].str.strip().str.title()

In [27]:
# 6. Data Types
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"] = df["TotalCharges"].fillna(df["TotalCharges"].median())


In [28]:
# 7. Outliers (IQR method)
for c in ["MonthlyCharges", "TotalCharges"]:
    q1, q3 = df[c].quantile([0.25, 0.75])
    iqr = q3 - q1
    df[c] = df[c].clip(q1 - 1.5 * iqr, q3 + 1.5 * iqr)

In [29]:
# 8. Feature engineering
df["AvgMonthlySpend"] = df["TotalCharges"] / (df["tenure"] + 1)
df["TenureGroup"] = pd.cut(df["tenure"], bins=[-1, 12, 24, 48, 72], labels=["0-1yr", "1-2yr", "2-4yr", "4-6"])

In [30]:
# 9. Encoding and scaling
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})
df = pd.get_dummies(df.drop("customerID", axis=1, errors="ignore"),
drop_first=True)
num_cols = ["tenure", "MonthlyCharges", "TotalCharges", "AvgMonthlySpend"]
df[num_cols] = StandardScaler().fit_transform(df[num_cols])

In [31]:
# 10. Train-test split
X = df.drop("Churn", axis=1)
y = df["Churn"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
 random_state=42, stratify=y)
print("Train:", X_train.shape, "Test:", X_test.shape)

Train: (5634, 34) Test: (1409, 34)


In [32]:
# 11. Export
df.to_csv("Telecom_Customer_Churn_Cleaned.csv", index=False)